# Causal Graph Operations

This notebook demonstrates causal graph construction, manipulation, and visualization using the datascienceutils library.

## Topics Covered:
- Creating causal graphs
- Adding nodes and edges
- Weighted edges
- Freezing edges/subgraphs (domain knowledge)
- Graph queries (parents, children)
- DOT export for Graphviz visualization
- Common causal structures

## Prerequisites
```bash
maturin develop --release
pip install jupyter
# Optional: install graphviz for rendering DOT files
# sudo apt-get install graphviz  # Linux
# brew install graphviz  # macOS
```

In [ ]:
import datascienceutils as dsu

print(f"Using datascienceutils v{dsu.__version__}")

## 1. Basic Graph Construction

Let's start by creating a simple causal graph.

In [ ]:
# Create a new causal graph
graph = dsu.CausalGraph()

# Add nodes
graph.add_node("Treatment")
graph.add_node("Outcome")
graph.add_node("Confounder")

# Add edges (directed: parent → child)
graph.add_edge("Confounder", "Treatment")
graph.add_edge("Confounder", "Outcome")
graph.add_edge("Treatment", "Outcome")

print(f"Graph created successfully!")
print(f"Number of nodes: {graph.num_nodes()}")
print(f"Number of edges: {graph.num_edges()}")

## 2. Graph Queries

Query the graph structure to understand relationships.

In [ ]:
# Query parents and children
print("Graph Structure:")
print(f"\nParents of 'Treatment': {graph.parents('Treatment')}")
print(f"Children of 'Treatment': {graph.children('Treatment')}")
print(f"\nParents of 'Outcome': {graph.parents('Outcome')}")
print(f"Children of 'Outcome': {graph.children('Outcome')}")
print(f"\nParents of 'Confounder': {graph.parents('Confounder')}")
print(f"Children of 'Confounder': {graph.children('Confounder')}")

## 3. DOT Export

Export the graph to DOT format for visualization with Graphviz.

In [ ]:
# Get DOT representation
dot_string = graph.to_dot()
print("DOT representation:")
print(dot_string)

# Save to file
graph.save_dot("simple_causal_graph.dot")
print("\nSaved to 'simple_causal_graph.dot'")
print("\nTo visualize:")
print("  dot -Tpng simple_causal_graph.dot -o simple_causal_graph.png")
print("  dot -Tsvg simple_causal_graph.dot -o simple_causal_graph.svg")

## 4. Weighted Edges

Add weights to edges to represent strength of causal relationships.

In [ ]:
# Create a new graph with weighted edges
weighted_graph = dsu.CausalGraph()

# Add nodes
weighted_graph.add_node("Education")
weighted_graph.add_node("Experience")
weighted_graph.add_node("Income")
weighted_graph.add_node("Age")

# Add weighted edges
weighted_graph.add_edge_weighted("Education", "Income", 0.8)  # Strong effect
weighted_graph.add_edge_weighted("Experience", "Income", 0.6)  # Moderate effect
weighted_graph.add_edge_weighted("Age", "Experience", 0.9)  # Strong effect
weighted_graph.add_edge_weighted("Age", "Income", 0.3)  # Weak direct effect

print(f"Weighted graph created!")
print(f"Nodes: {weighted_graph.num_nodes()}")
print(f"Edges: {weighted_graph.num_edges()}")

# Export
weighted_graph.save_dot("weighted_causal_graph.dot")
print("\nSaved to 'weighted_causal_graph.dot'")

## 5. Freezing Edges (Domain Knowledge)

Freeze edges or subgraphs to incorporate domain knowledge and prevent modifications.

In [ ]:
# Create a graph with domain knowledge
medical_graph = dsu.CausalGraph()

# Add nodes
medical_graph.add_node("Smoking")
medical_graph.add_node("Exercise")
medical_graph.add_node("Diet")
medical_graph.add_node("HeartDisease")
medical_graph.add_node("Age")

# Add edges
medical_graph.add_edge("Age", "HeartDisease")
medical_graph.add_edge("Smoking", "HeartDisease")
medical_graph.add_edge("Exercise", "HeartDisease")
medical_graph.add_edge("Diet", "HeartDisease")

# Freeze known causal relationships (domain knowledge)
# We know for certain that smoking causes heart disease
medical_graph.freeze_edge("Smoking", "HeartDisease")

# Freeze a subgraph of well-established relationships
medical_graph.freeze_subgraph(["Age", "HeartDisease"])

print("Medical causal graph created with frozen edges!")
print(f"Nodes: {medical_graph.num_nodes()}")
print(f"Edges: {medical_graph.num_edges()}")

# Export
medical_graph.save_dot("medical_causal_graph.dot")
print("\nSaved to 'medical_causal_graph.dot'")

## 6. Common Causal Structures

Let's create examples of common causal structures.

### 6.1 Confounding (Fork)

A confounder affects both treatment and outcome.

In [ ]:
# Confounding structure: C → T, C → Y
confounding = dsu.CausalGraph()
confounding.add_node("Confounder")
confounding.add_node("Treatment")
confounding.add_node("Outcome")
confounding.add_edge("Confounder", "Treatment")
confounding.add_edge("Confounder", "Outcome")
confounding.add_edge("Treatment", "Outcome")

confounding.save_dot("confounding_structure.dot")
print("Confounding structure:")
print(confounding.to_dot())
print("\nSaved to 'confounding_structure.dot'")

### 6.2 Mediation (Chain)

Treatment affects outcome through a mediator.

In [ ]:
# Mediation structure: T → M → Y
mediation = dsu.CausalGraph()
mediation.add_node("Treatment")
mediation.add_node("Mediator")
mediation.add_node("Outcome")
mediation.add_edge("Treatment", "Mediator")
mediation.add_edge("Mediator", "Outcome")
mediation.add_edge("Treatment", "Outcome")  # Direct effect

mediation.save_dot("mediation_structure.dot")
print("Mediation structure:")
print(mediation.to_dot())
print("\nSaved to 'mediation_structure.dot'")

### 6.3 Collider (Inverted Fork)

Both treatment and outcome affect a collider.

In [ ]:
# Collider structure: T → C, Y → C
collider = dsu.CausalGraph()
collider.add_node("Treatment")
collider.add_node("Outcome")
collider.add_node("Collider")
collider.add_edge("Treatment", "Collider")
collider.add_edge("Outcome", "Collider")
collider.add_edge("Treatment", "Outcome")

collider.save_dot("collider_structure.dot")
print("Collider structure:")
print(collider.to_dot())
print("\nSaved to 'collider_structure.dot'")
print("\nNote: Conditioning on a collider creates spurious association!")

### 6.4 Instrumental Variable

An instrument affects treatment but not outcome directly.

In [ ]:
# IV structure: I → T → Y, U → T, U → Y
iv_graph = dsu.CausalGraph()
iv_graph.add_node("Instrument")
iv_graph.add_node("Treatment")
iv_graph.add_node("Outcome")
iv_graph.add_node("Unobserved")

iv_graph.add_edge("Instrument", "Treatment")
iv_graph.add_edge("Treatment", "Outcome")
iv_graph.add_edge("Unobserved", "Treatment")
iv_graph.add_edge("Unobserved", "Outcome")

iv_graph.save_dot("iv_structure.dot")
print("Instrumental Variable structure:")
print(iv_graph.to_dot())
print("\nSaved to 'iv_structure.dot'")
print("\nIV assumptions:")
print("  1. Instrument affects treatment (relevance)")
print("  2. Instrument doesn't affect outcome directly (exclusion)")
print("  3. Instrument is independent of unobserved confounders")

## 7. Complex Real-World Example

Let's create a more complex causal graph for a real-world scenario.

In [ ]:
# Marketing campaign effectiveness
marketing = dsu.CausalGraph()

# Add all nodes
nodes = [
    "CustomerAge",
    "Income",
    "PreviousPurchases",
    "EmailCampaign",
    "WebsiteVisits",
    "ProductViews",
    "Purchase",
    "Satisfaction",
    "Repeat"
]

for node in nodes:
    marketing.add_node(node)

# Add causal relationships
# Demographics affect behavior
marketing.add_edge("CustomerAge", "Income")
marketing.add_edge("Income", "PreviousPurchases")
marketing.add_edge("CustomerAge", "EmailCampaign")  # Targeting
marketing.add_edge("Income", "EmailCampaign")  # Targeting

# Campaign effects
marketing.add_edge("EmailCampaign", "WebsiteVisits")
marketing.add_edge("WebsiteVisits", "ProductViews")
marketing.add_edge("ProductViews", "Purchase")

# Previous behavior affects current
marketing.add_edge("PreviousPurchases", "WebsiteVisits")
marketing.add_edge("PreviousPurchases", "Purchase")

# Purchase outcomes
marketing.add_edge("Purchase", "Satisfaction")
marketing.add_edge("Satisfaction", "Repeat")
marketing.add_edge("Purchase", "Repeat")

# Income affects purchase ability
marketing.add_edge("Income", "Purchase")

print(f"Marketing causal graph created!")
print(f"Nodes: {marketing.num_nodes()}")
print(f"Edges: {marketing.num_edges()}")

# Query key relationships
print(f"\nWhat affects Purchase?")
print(f"  Parents: {marketing.parents('Purchase')}")
print(f"\nWhat does EmailCampaign affect?")
print(f"  Children: {marketing.children('EmailCampaign')}")

# Export
marketing.save_dot("marketing_causal_graph.dot")
print("\nSaved to 'marketing_causal_graph.dot'")

## 8. Graph Analysis Tips

Here are some tips for working with causal graphs:

In [ ]:
# Create a graph for demonstration
demo = dsu.CausalGraph()

# Add nodes
for node in ["A", "B", "C", "D", "E"]:
    demo.add_node(node)

# Create a path: A → B → C → D
demo.add_edge("A", "B")
demo.add_edge("B", "C")
demo.add_edge("C", "D")

# Add a confounder: E → B, E → D
demo.add_edge("E", "B")
demo.add_edge("E", "D")

print("Graph Analysis:")
print("\n1. Identify confounders:")
print(f"   Parents of B: {demo.parents('B')} (A and E are potential confounders)")
print(f"   Parents of D: {demo.parents('D')} (C and E are potential confounders)")

print("\n2. Trace causal paths:")
print(f"   A → B: Direct")
print(f"   A → B → C → D: Indirect through B and C")

print("\n3. Identify backdoor paths:")
print(f"   B ← E → D: Backdoor path between B and D")
print(f"   Need to condition on E to block this path")

demo.save_dot("demo_graph.dot")
print("\nSaved to 'demo_graph.dot'")

## Summary

This notebook demonstrated:

1. **Basic Operations**:
   - Creating graphs
   - Adding nodes and edges
   - Querying graph structure

2. **Advanced Features**:
   - Weighted edges for effect strength
   - Freezing edges to incorporate domain knowledge
   - DOT export for visualization

3. **Common Structures**:
   - Confounding (fork)
   - Mediation (chain)
   - Collider (inverted fork)
   - Instrumental variables

4. **Real-World Applications**:
   - Medical causal graphs
   - Marketing effectiveness
   - Complex multi-variable systems

### Best Practices:

- **Start simple**: Begin with core variables and add complexity
- **Use domain knowledge**: Freeze well-established relationships
- **Document assumptions**: Use weights to indicate confidence
- **Visualize**: Export to DOT and render with Graphviz
- **Iterate**: Refine graph based on data and expert feedback

### Visualization Commands:

```bash
# Generate PNG
dot -Tpng graph.dot -o graph.png

# Generate SVG (scalable)
dot -Tsvg graph.dot -o graph.svg

# Generate PDF
dot -Tpdf graph.dot -o graph.pdf

# Use different layouts
dot -Tpng -Kneato graph.dot -o graph_neato.png  # Force-directed
dot -Tpng -Kcirco graph.dot -o graph_circo.png  # Circular
```

### Next Steps:

- Apply causal inference methods to your graphs
- Use graphs to identify confounders and mediators
- Validate graph structure with data
- Explore causal discovery algorithms